# Codigo para generar mapas de Riesgos

map_isosuperficies.py

Genera un mapa (contourf / isosuperficies) por cada columna (empresa) en un archivo CSV que
contiene coordenadas "x_km,y_km" (en kilómetros, UTM zona 14) y valores de riesgo (riesgo cancer
lifetime). Convierte coordenadas UTM (en km) a metros y luego a lon/lat (EPSG:4326) para graficar.

Características:
- Interpolación con scipy.interpolate.griddata (cubic -> linear -> nearest fallback).
- Control de transparencia (alpha) y número de niveles.
- Guarda una figura PNG por columna en una carpeta de salida.
- Permite seleccionar columnas, resolución de rejilla (m), colormap, EPSG UTM (por defecto 32614).




Uso:
    python map_isosuperficies.py --csv riesgos.csv --outdir figuras --alpha 0.6 --grid_res 250 --nlevels 12

Supuestos:
- Coordenadas x_km,y_km están en kilómetros. Se multiplican por 1000 para pasar a metros.
- UTM zona 14 Norte por defecto (EPSG:32614). Si sus datos están en hemisferio sur, use EPSG 32714.

Requisitos:
    pip install pandas numpy matplotlib scipy pyproj

In [ ]:
%%bash
python mapv3_iso.py --csv riesgos.csv --outdir figuras --alpha 0.4 --grid_res 250 --nlevels 12 --columns Total

# Esta es la version que genera mapas en Jupyter
Los mapas generados se presentan en el cuaderno

## Version para guardar y poner los nombres de las empresas
Lee riesgos.csv y genera mapas con mapa base de fondo

- Entradas:
    -  riesgos.csv
    -  empresas.csv
- Salida:
    - map_{CLAVE}.png

In [ ]:
# Lee riesgo y genera mapas con mapa base de fondo
# entradas:
#          riesgos.csv y riesgosC.csv 
#          empresas.csv
# Salida:
#         map_{CLAVE}.png
#
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib import colormaps
import matplotlib.cm as cm
from scipy.interpolate import griddata
import contextily as ctx
from pyproj import Transformer

# --- Parámetros ---
csv_file = 'riesgosC.csv'
empresas_file = 'empresas.csv'  # Archivo con claves y nombres de empresas

utm_zone = 14  # Zona UTM 14N
epsg_utm = f"EPSG:326{utm_zone:02d}"  # EPSG código para UTM zona 14N (hemisferio norte)
epsg_wgs84 = "EPSG:4326"
epsg_3857 = "EPSG:3857"

alpha_default = 0.4  # Transparencia default para las isosuperficies

# Rango fijo para la grilla en lon/lat según tu petición:
lon_min, lon_max = -1.120, -1.113
lat_min, lat_max = 2.92, 3.00

# --- Leer datos ---
df = pd.read_csv(csv_file)

# Leer archivo de empresas y crear diccionario clave -> nombre
df_empresas = pd.read_csv(empresas_file)
clave_a_nombre = dict(zip(df_empresas['CLAVE'].str.strip(), df_empresas['Nombre'].str.strip()))

# Extraer coordenadas UTM (km a metros)
x_utm = df['x_km'].values * 1000
y_utm = df['y_km'].values * 1000

# Transformar UTM -> WGS84 (lon, lat)
transformer_utm_to_wgs84 = Transformer.from_crs(epsg_utm, epsg_wgs84, always_xy=True)
lon, lat = transformer_utm_to_wgs84.transform(x_utm, y_utm)

# Agregar lon/lat al dataframe por si quieres usarlo luego
df['lon'] = lon
df['lat'] = lat
# 
lon_min, lon_max = lon.min(), lon.max()
lat_min, lat_max = lat.min(), lat.max()
# Transformar lon/lat -> Web Mercator EPSG:3857 para graficar con contextily
transformer_wgs84_to_3857 = Transformer.from_crs(epsg_wgs84, epsg_3857, always_xy=True)
merc_x, merc_y = transformer_wgs84_to_3857.transform(lon, lat)

# Columnas de empresas + Total (excluyendo coordenadas)
cols_empresas = [col for col in df.columns if col not in ['x_km', 'y_km', 'lon', 'lat']]

# Crear una grilla regular en lon/lat para interpolación con rango fijo solicitado
num_grid_points = 300

grid_lon = np.linspace(lon_min, lon_max, num_grid_points)
grid_lat = np.linspace(lat_min, lat_max, num_grid_points)
grid_lon_mesh, grid_lat_mesh = np.meshgrid(grid_lon, grid_lat)

# Transformar la grilla a mercator para graficar
grid_x_merc, grid_y_merc = transformer_wgs84_to_3857.transform(
    grid_lon_mesh.flatten(), grid_lat_mesh.flatten())
grid_x_merc = np.array(grid_x_merc).reshape(grid_lon_mesh.shape)
grid_y_merc = np.array(grid_y_merc).reshape(grid_lat_mesh.shape)

def plot_risk_surface(company_col, alpha=alpha_default):
    values = df[company_col].values
    
    # Determinar el umbral según la columna
    es_total = company_col.strip().lower() == 'total'
    umbral = 10 if es_total else 1e-6
    
    # Convertir a float para poder trabajar con NaN
    values_float = values.astype(float)
    
    # Verificar que hay valores por encima del umbral
    valores_validos = values_float >= umbral
    if not np.any(valores_validos):
        print(f"  Advertencia: No hay valores >= {umbral} para {company_col}. No se genera mapa.")
        return
    
    # Interpolación con TODOS los valores (sin filtrar antes)
    grid_values = griddata(
        points=(lon, lat),
        values=values_float,
        xi=(grid_lon_mesh, grid_lat_mesh),
        method='linear'
    )
    
    # DESPUÉS de interpolar, aplicar máscara para valores bajo el umbral
    grid_values_masked = grid_values.copy()
    grid_values_masked[grid_values < umbral] = np.nan
    
    # ---- Crear colormap con transparencia en valores bajo el umbral ----
    base_cmap = colormaps.get_cmap('inferno').resampled(256)        # 256 colores
    colors = base_cmap(np.linspace(0, 1, 256))     # Copia del array RGBA
    
    # Cambiar la transparencia en el color más bajo
    colors[0, -1] = 0.0    # alpha = 0 para el primer color
    
    # Crear el colormap modificado
    cmap_mod = ListedColormap(colors)

    # Definir niveles (solo para valores válidos >= umbral)
    vmin = np.nanmin(grid_values_masked)
    vmax = np.nanmax(grid_values_masked)
    
    # Si todos los valores interpolados son NaN, no generar el mapa
    if np.isnan(vmin) or np.isnan(vmax):
        print(f"  Advertencia: Después de interpolar, no hay valores válidos >= {umbral} para {company_col}. No se genera mapa.")
        return
    
    # Asegurar que vmin sea al menos el umbral
    vmin = max(vmin, umbral)
    levels = np.linspace(vmin, vmax, 15)

    # Norma para asignar colores a intervalos
    norm = BoundaryNorm(levels, ncolors=cmap_mod.N, clip=True)
    # -----------------------------------------------------------

    fig, ax = plt.subplots(figsize=(10,10))

    contourf = ax.contourf(
        grid_x_merc,
        grid_y_merc,
        grid_values_masked,
        levels=levels,
        cmap=cmap_mod,
        norm=norm,
        alpha=alpha
    )

    cbar = plt.colorbar(contourf, ax=ax)

    # Etiquetas y títulos
    if es_total:
        nombre_empresa_titulo = "Riesgo Total"
        cbar.set_label('Riesgo a cáncer - Riesgo Total (valores > 1)')
        filename_name_part = "Total"
    else:
        nombre_empresa_titulo = clave_a_nombre.get(company_col.strip(), company_col)
        cbar.set_label(f'Riesgo a cáncer - {nombre_empresa_titulo} (valores > 1e-6)')
        filename_name_part = company_col.strip()

    ax.set_title(f'Mapa de Riesgo a Cáncer - por: {nombre_empresa_titulo}')
    ax.set_xlabel('Coordenada X (Web Mercator)')
    ax.set_ylabel('Coordenada Y (Web Mercator)')

    margin_x = (merc_x.max() - merc_x.min()) * 0.01
    margin_y = (merc_y.max() - merc_y.min()) * 0.01
    ax.set_xlim(merc_x.min() - margin_x, merc_x.max() + margin_x)
    ax.set_ylim(merc_y.min() - margin_y, merc_y.max() + margin_y)

    try:
        ctx.add_basemap(ax,
                        source=ctx.providers.OpenStreetMap.Mapnik,
                        zoom=12)
    except Exception as e:
        print("Error al cargar mapa base:", e)

    plt.legend().remove()
    plt.tight_layout()

    safe_name_part = "".join(c for c in filename_name_part if c.isalnum() or c in ('_',)).rstrip()
    filename = f'map_{safe_name_part}.png'
    
    plt.savefig(filename, dpi=300)
    print(f"Guardado mapa en {filename}")
    plt.close()

# Generar mapas para todas las columnas incluyendo Total con transparencia ajustable
for empresa in cols_empresas:
    print(f"Generando mapa para {empresa}...")
    plot_risk_surface(empresa, alpha=0.45)  # Cambia alpha entre 0 y 1 si quieres

print("Todos los mapas han sido generados y guardados.")

## Para NO Cancer
valor umbral 0.50

In [ ]:
# Lee riesgo y genera mapas con mapa base de fondo
# Para NO CANCER
# entradas:
#          ncancerI.csv ncancerC.csv
#          empresas.csv
# Salida:
#         mapNC_{CLAVE}.png
#
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib import colormaps
import matplotlib.cm as cm
from scipy.interpolate import griddata
import contextily as ctx
from pyproj import Transformer

# --- Parámetros ---
# Primero para empresas ncancerI y luego por sustancia ncancerC
csv_file = 'ncancerC.csv'
empresas_file = 'empresas.csv'  # Archivo con claves y nombres de empresas

utm_zone = 14  # Zona UTM 14N
epsg_utm = f"EPSG:326{utm_zone:02d}"  # EPSG código para UTM zona 14N (hemisferio norte)
epsg_wgs84 = "EPSG:4326"
epsg_3857 = "EPSG:3857"

alpha_default = 0.4  # Transparencia default para las isosuperficies

# Rango fijo para la grilla en lon/lat según tu petición:
lon_min, lon_max = -1.120, -1.113
lat_min, lat_max = 2.92, 3.00

# --- Leer datos ---
df = pd.read_csv(csv_file)

# Leer archivo de empresas y crear diccionario clave -> nombre
df_empresas = pd.read_csv(empresas_file)
clave_a_nombre = dict(zip(df_empresas['CLAVE'].str.strip(), df_empresas['Nombre'].str.strip()))

# Extraer coordenadas UTM (km a metros)
x_utm = df['x_km'].values * 1000
y_utm = df['y_km'].values * 1000

# Transformar UTM -> WGS84 (lon, lat)
transformer_utm_to_wgs84 = Transformer.from_crs(epsg_utm, epsg_wgs84, always_xy=True)
lon, lat = transformer_utm_to_wgs84.transform(x_utm, y_utm)

# Agregar lon/lat al dataframe por si quieres usarlo luego
df['lon'] = lon
df['lat'] = lat
# 
lon_min, lon_max = lon.min(), lon.max()
lat_min, lat_max = lat.min(), lat.max()
# Transformar lon/lat -> Web Mercator EPSG:3857 para graficar con contextily
transformer_wgs84_to_3857 = Transformer.from_crs(epsg_wgs84, epsg_3857, always_xy=True)
merc_x, merc_y = transformer_wgs84_to_3857.transform(lon, lat)

# Columnas de empresas + Total (excluyendo coordenadas)
cols_empresas = [col for col in df.columns if col not in ['x_km', 'y_km', 'lon', 'lat']]

# Crear una grilla regular en lon/lat para interpolación con rango fijo solicitado
num_grid_points = 300

grid_lon = np.linspace(lon_min, lon_max, num_grid_points)
grid_lat = np.linspace(lat_min, lat_max, num_grid_points)
grid_lon_mesh, grid_lat_mesh = np.meshgrid(grid_lon, grid_lat)

# Transformar la grilla a mercator para graficar
grid_x_merc, grid_y_merc = transformer_wgs84_to_3857.transform(
    grid_lon_mesh.flatten(), grid_lat_mesh.flatten())
grid_x_merc = np.array(grid_x_merc).reshape(grid_lon_mesh.shape)
grid_y_merc = np.array(grid_y_merc).reshape(grid_lat_mesh.shape)

def plot_risk_surface(company_col, alpha=alpha_default):
    values = df[company_col].values
    
    # Determinar el umbral según la columna
    es_total = company_col.strip().lower() == 'total'
    umbral = 0.5 if es_total else 0.25
    
    # Convertir a float para poder trabajar con NaN
    values_float = values.astype(float)
    
    # Verificar que hay valores por encima del umbral
    valores_validos = values_float >= umbral
    if not np.any(valores_validos):
        print(f"  Advertencia: No hay valores >= {umbral} para {company_col}. No se genera mapa.")
        return
    
    # Interpolación con TODOS los valores (sin filtrar antes)
    grid_values = griddata(
        points=(lon, lat),
        values=values_float,
        xi=(grid_lon_mesh, grid_lat_mesh),
        method='linear'
    )
    
    # DESPUÉS de interpolar, aplicar máscara para valores bajo el umbral
    grid_values_masked = grid_values.copy()
    grid_values_masked[grid_values < umbral] = np.nan
    
    # ---- Crear colormap con transparencia en valores bajo el umbral ----
    base_cmap = colormaps.get_cmap('inferno').resampled(256)        # 256 colores
    colors = base_cmap(np.linspace(0, 1, 256))     # Copia del array RGBA
    
    # Cambiar la transparencia en el color más bajo
    colors[0, -1] = 0.0    # alpha = 0 para el primer color
    
    # Crear el colormap modificado
    cmap_mod = ListedColormap(colors)

    # Definir niveles (solo para valores válidos >= umbral)
    vmin = np.nanmin(grid_values_masked)
    vmax = np.nanmax(grid_values_masked)
    
    # Si todos los valores interpolados son NaN, no generar el mapa
    if np.isnan(vmin) or np.isnan(vmax):
        print(f"  Advertencia: Después de interpolar, no hay valores válidos >= {umbral} para {company_col}. No se genera mapa.")
        return
    
    # Asegurar que vmin sea al menos el umbral
    vmin = max(vmin, umbral)
    levels = np.linspace(vmin, vmax, 15)

    # Norma para asignar colores a intervalos
    norm = BoundaryNorm(levels, ncolors=cmap_mod.N, clip=True)
    # -----------------------------------------------------------

    fig, ax = plt.subplots(figsize=(10,10))

    contourf = ax.contourf(
        grid_x_merc,
        grid_y_merc,
        grid_values_masked,
        levels=levels,
        cmap=cmap_mod,
        norm=norm,
        alpha=alpha
    )

    cbar = plt.colorbar(contourf, ax=ax)

    # Etiquetas y títulos
    if es_total:
        nombre_empresa_titulo = "Riesgo Total"
        cbar.set_label('Riesgo No cáncer - Total (valores > 1)')
        filename_name_part = "Total"
    else:
        nombre_empresa_titulo = clave_a_nombre.get(company_col.strip(), company_col)
        cbar.set_label(f'Riesgo No cáncer - {nombre_empresa_titulo} (valores > 1e-6)')
        filename_name_part = company_col.strip()

    ax.set_title(f'Mapa de Riesgo No Cáncer - por: {nombre_empresa_titulo}')
    ax.set_xlabel('Coordenada X (Web Mercator)')
    ax.set_ylabel('Coordenada Y (Web Mercator)')

    margin_x = (merc_x.max() - merc_x.min()) * 0.01
    margin_y = (merc_y.max() - merc_y.min()) * 0.01
    ax.set_xlim(merc_x.min() - margin_x, merc_x.max() + margin_x)
    ax.set_ylim(merc_y.min() - margin_y, merc_y.max() + margin_y)

    try:
        ctx.add_basemap(ax,
                        source=ctx.providers.OpenStreetMap.Mapnik,
                        zoom=12)
    except Exception as e:
        print("Error al cargar mapa base:", e)

    plt.legend().remove()
    plt.tight_layout()

    safe_name_part = "".join(c for c in filename_name_part if c.isalnum() or c in ('_',)).rstrip()
    filename = f'mapNC_{safe_name_part}.png'
    
    plt.savefig(filename, dpi=300)
    print(f"Guardado mapa en {filename}")
    plt.close()

# Generar mapas para todas las columnas incluyendo Total con transparencia ajustable
for empresa in cols_empresas:
    print(f"Generando mapa para {empresa}...")
    plot_risk_surface(empresa, alpha=0.45)  # Cambia alpha entre 0 y 1 si quieres

print("Todos los mapas han sido generados y guardados.")

## Crea documento a partir de las figuras
Este código crea:

+ Empresas.docx
  
	-	Solo imágenes cuyas claves aparecen en empresas.csv.
	-	Título principal.
	-	Índice automático.
	-	Secciones por empresa.
	-	Tamaño reducido de imagen.

+ Sustancias.docx
  
	-	Contiene todas las demás imágenes.
	-	El mismo formato (TOC, secciones, reducción de tamaño).
## Para cáncer

In [ ]:
import os
import pandas as pd
from PIL import Image
from docx import Document
from docx.shared import Inches
from docx.enum.section import WD_SECTION_START
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# ===============================================
#  FUNCIONES AUXILIARES
# ===============================================

def agregar_TOC(doc):
    paragraph = doc.add_paragraph()
    run = paragraph.add_run()
    fld_simple = OxmlElement('w:fldSimple')
    fld_simple.set(qn('w:instr'), 'TOC \\o "1-3" \\h \\z \\u')
    run._r.append(fld_simple)


def comprimir_imagen(ruta_original, nombre_tmp="tmp_img.jpg",
                     calidad=60, max_width_px=1200):
    """
    Comprime la imagen PNG para reducir su peso:
    - Reduce resolución si es muy grande
    - Convierte a JPEG con calidad inferior (peso reducido)
    """
    try:
        img = Image.open(ruta_original).convert("RGB")

        # Redimensionar si es muy grande
        w, h = img.size
        if w > max_width_px:
            factor = max_width_px / float(w)
            img = img.resize((int(w * factor), int(h * factor)), Image.LANCZOS)

        # Guardar comprimido
        tmp_path = nombre_tmp
        img.save(tmp_path, "JPEG", quality=calidad, optimize=True)
        return tmp_path

    except Exception as e:
        print("Error comprimiendo imagen:", ruta_original, e)
        return ruta_original


def establecer_dos_columnas(documento):
    """
    Aplica formato a dos columnas al documento, a partir de la sección actual.
    """
    sectPr = documento.sections[-1]._sectPr
    cols = sectPr.xpath("./w:cols")
    
    if cols:
        cols = cols[0]
    else:
        cols = OxmlElement("w:cols")
        sectPr.append(cols)

    cols.set(qn("w:num"), "2")  # número de columnas = 2


# ===============================================
#  LEER ARCHIVO DE EMPRESAS
# ===============================================
df_empresas = pd.read_csv("empresas.csv")
df_empresas['CLAVE'] = df_empresas['CLAVE'].str.strip()
df_empresas['Nombre'] = df_empresas['Nombre'].str.strip()

claves_empresas = set(df_empresas['CLAVE'].tolist())
clave_a_nombre = dict(zip(df_empresas['CLAVE'], df_empresas['Nombre']))

# ===============================================
#  LEER IMÁGENES DE DIRECTORIO
# ===============================================
imagenes = sorted([
    f for f in os.listdir(".")
    if f.lower().endswith(".png") and f.startswith("map_")
])

imgs_empresas = []
imgs_sustancias = []

for img in imagenes:
    clave = img.replace("map_", "").replace(".png", "").strip()
    
    if clave in claves_empresas:
        imgs_empresas.append(img)
    else:
        imgs_sustancias.append(img)


# ===============================================
#  FUNCIÓN PRINCIPAL PARA CREAR DOCUMENTOS
# ===============================================
def crear_documento(nombre_doc, lista_imagenes, es_empresa=True):
    if not lista_imagenes:
        print(f"No hay imágenes para: {nombre_doc}")
        return

    doc = Document()

    # ---------------------------
    # PORTADA
    # ---------------------------
    titulo = "Mapas de Riesgo - Empresas" if es_empresa else "Mapas de Riesgo - Sustancias"
    doc.add_heading(titulo, level=0)
    doc.add_page_break()

    # ---------------------------
    # ÍNDICE
    # ---------------------------
    doc.add_heading("Índice", level=1)
    agregar_TOC(doc)
    doc.add_page_break()

    # ---------------------------
    # ESTABLECER DOS COLUMNAS
    # ---------------------------
    #establecer_dos_columnas(doc)

    # ---------------------------
    # CONTENIDO (IMÁGENES)
    # ---------------------------
    for img in lista_imagenes:
        clave = img.replace("map_", "").replace(".png", "").strip()
        nombre = clave_a_nombre.get(clave, clave) if es_empresa else clave

        # Título de sección
        doc.add_heading(nombre, level=1)

        # Comprimir imagen temporalmente
        img_tmp = comprimir_imagen(img)

        # Insertar en Word (ya comprimida)
        doc.add_picture(img_tmp, width=Inches(4))

    # Guardar
    doc.save(nombre_doc)
    print(f"Documento generado: {nombre_doc}")


# ===============================================
#  GENERAR DOCUMENTOS
# ===============================================
crear_documento("EmpresasCancer.docx", imgs_empresas, es_empresa=True)
crear_documento("SustanciasCancer.docx", imgs_sustancias, es_empresa=False)

## Para no-cáncer
Se tiene que correr ncancerI y ncancerC

In [ ]:
import os
import pandas as pd
from PIL import Image
from docx import Document
from docx.shared import Inches
from docx.enum.section import WD_SECTION_START
from docx.oxml import OxmlElement
from docx.oxml.ns import qn

# ===============================================
#  FUNCIONES AUXILIARES
# ===============================================

def agregar_TOC(doc):
    paragraph = doc.add_paragraph()
    run = paragraph.add_run()
    fld_simple = OxmlElement('w:fldSimple')
    fld_simple.set(qn('w:instr'), 'TOC \\o "1-3" \\h \\z \\u')
    run._r.append(fld_simple)


def comprimir_imagen(ruta_original, nombre_tmp="tmp_img.jpg",
                     calidad=60, max_width_px=1200):
    """
    Comprime la imagen PNG para reducir su peso:
    - Reduce resolución si es muy grande
    - Convierte a JPEG con calidad inferior (peso reducido)
    """
    try:
        img = Image.open(ruta_original).convert("RGB")

        # Redimensionar si es muy grande
        w, h = img.size
        if w > max_width_px:
            factor = max_width_px / float(w)
            img = img.resize((int(w * factor), int(h * factor)), Image.LANCZOS)

        # Guardar comprimido
        tmp_path = nombre_tmp
        img.save(tmp_path, "JPEG", quality=calidad, optimize=True)
        return tmp_path

    except Exception as e:
        print("Error comprimiendo imagen:", ruta_original, e)
        return ruta_original


def establecer_dos_columnas(documento):
    """
    Aplica formato a dos columnas al documento, a partir de la sección actual.
    """
    sectPr = documento.sections[-1]._sectPr
    cols = sectPr.xpath("./w:cols")
    
    if cols:
        cols = cols[0]
    else:
        cols = OxmlElement("w:cols")
        sectPr.append(cols)

    cols.set(qn("w:num"), "2")  # número de columnas = 2


# ===============================================
#  LEER ARCHIVO DE EMPRESAS
# ===============================================
df_empresas = pd.read_csv("empresas.csv")
df_empresas['CLAVE'] = df_empresas['CLAVE'].str.strip()
df_empresas['Nombre'] = df_empresas['Nombre'].str.strip()

claves_empresas = set(df_empresas['CLAVE'].tolist())
clave_a_nombre = dict(zip(df_empresas['CLAVE'], df_empresas['Nombre']))

# ===============================================
#  LEER IMÁGENES DE DIRECTORIO
# ===============================================
imagenes = sorted([
    f for f in os.listdir(".")
    if f.lower().endswith(".png") and f.startswith("mapNC_")
])

imgs_empresas = []
imgs_sustancias = []

for img in imagenes:
    clave = img.replace("mapNC_", "").replace(".png", "").strip()
    
    if clave in claves_empresas:
        imgs_empresas.append(img)
    else:
        imgs_sustancias.append(img)


# ===============================================
#  FUNCIÓN PRINCIPAL PARA CREAR DOCUMENTOS
# ===============================================
def crear_documento(nombre_doc, lista_imagenes, es_empresa=True):
    if not lista_imagenes:
        print(f"No hay imágenes para: {nombre_doc}")
        return

    doc = Document()

    # ---------------------------
    # PORTADA
    # ---------------------------
    titulo = "Mapas de Efectos no cáncer - Empresas" if es_empresa else "Mapas de Efectos no cáncer - Sustancias"
    doc.add_heading(titulo, level=0)
    doc.add_page_break()

    # ---------------------------
    # ÍNDICE
    # ---------------------------
    doc.add_heading("Índice", level=1)
    agregar_TOC(doc)
    doc.add_page_break()

    # ---------------------------
    # ESTABLECER DOS COLUMNAS
    # ---------------------------
    #establecer_dos_columnas(doc)

    # ---------------------------
    # CONTENIDO (IMÁGENES)
    # ---------------------------
    for img in lista_imagenes:
        clave = img.replace("mapNC_", "").replace(".png", "").strip()
        nombre = clave_a_nombre.get(clave, clave) if es_empresa else clave

        # Título de sección
        doc.add_heading(nombre, level=1)

        # Comprimir imagen temporalmente
        img_tmp = comprimir_imagen(img)

        # Insertar en Word (ya comprimida)
        doc.add_picture(img_tmp, width=Inches(4))

    # Guardar
    doc.save(nombre_doc)
    print(f"Documento generado: {nombre_doc}")


# ===============================================
#  GENERAR DOCUMENTOS
# ===============================================
crear_documento("EmpresasNC.docx", imgs_empresas, es_empresa=True)
crear_documento("SustanciasNC.docx", imgs_sustancias, es_empresa=False)

# Genera archivos KMZ
- Lee los mismos datos que crea_mapa5.py para obtener las coordenadas exactas
- Genera archivos KMZ con las coordenadas correctas (sin márgenes ni barras de color)

In [ ]:
# Genera UN SOLO archivo KMZ con TODAS las capas de riesgo
# Cada empresa/Total será una capa individual que puede activarse/desactivarse en Google Earth
# Para crear mapas Cancer
import pandas as pd
import numpy as np
from pyproj import Transformer
import zipfile
import os
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib import colormaps

# --- Parámetros (deben coincidir con crea_mapa5.py) ---
csv_file = 'ncancerC.csv'
empresas_file = 'empresas.csv'
output_kmz = 'mapas_riesgoNC.kmz'

utm_zone = 14
epsg_utm = f"EPSG:326{utm_zone:02d}"
epsg_wgs84 = "EPSG:4326"

# --- Leer datos ---
df = pd.read_csv(csv_file)
df_empresas = pd.read_csv(empresas_file)

# Crear diccionario clave -> nombre
clave_a_nombre = dict(zip(df_empresas['CLAVE'].str.strip(), df_empresas['Nombre'].str.strip()))

# Extraer coordenadas UTM (km a metros)
x_utm = df['x_km'].values * 1000
y_utm = df['y_km'].values * 1000

# Transformar UTM -> WGS84 (lon, lat)
transformer_utm_to_wgs84 = Transformer.from_crs(epsg_utm, epsg_wgs84, always_xy=True)
lon, lat = transformer_utm_to_wgs84.transform(x_utm, y_utm)

# Obtener límites reales de los datos
lon_min, lon_max = lon.min(), lon.max()
lat_min, lat_max = lat.min(), lat.max()

# Columnas de empresas + Total
cols_empresas = [col for col in df.columns if col not in ['x_km', 'y_km', 'lon', 'lat']]

def generate_overlay_image(company_col, output_png, width=2000, height=2000):
    """
    Genera una imagen PNG sin márgenes ni barras de color, solo la superficie de riesgo
    Retorna: (success, vmin, vmax, nombre_empresa, clave, es_total)
    """
    values = df[company_col].values
    
    # Determinar si es Total y obtener nombre
    es_total = company_col.strip().lower() == 'total'
    
    if es_total:
        nombre_empresa = "Riesgo Total"
        clave = None
        umbral = 1.0
    else:
        clave = company_col.strip()
        if clave in clave_a_nombre:
            nombre_empresa = clave_a_nombre[clave]
        else:
            nombre_empresa = clave
            print(f"  Advertencia: Clave '{clave}' no encontrada en empresas.csv")
        umbral = 0.25
    
    # Convertir a float para poder trabajar con NaN
    values_float = values.astype(float)
    
    # Verificar que hay valores por encima del umbral
    valores_validos = values_float >= umbral
    if not np.any(valores_validos):
        print(f"  Advertencia: No hay valores >= {umbral} para {nombre_empresa}")
        return None
    
    # Crear grilla para interpolación
    num_grid_points = 400
    grid_lon = np.linspace(lon_min, lon_max, num_grid_points)
    grid_lat = np.linspace(lat_min, lat_max, num_grid_points)
    grid_lon_mesh, grid_lat_mesh = np.meshgrid(grid_lon, grid_lat)
    
    # Interpolación con TODOS los valores
    grid_values = griddata(
        points=(lon, lat),
        values=values_float,
        xi=(grid_lon_mesh, grid_lat_mesh),
        method='linear'
    )
    
    # Aplicar máscara para valores bajo el umbral
    grid_values_masked = grid_values.copy()
    grid_values_masked[grid_values < umbral] = np.nan
    
    # Verificar valores válidos después de interpolar
    vmin = np.nanmin(grid_values_masked)
    vmax = np.nanmax(grid_values_masked)
    
    if np.isnan(vmin) or np.isnan(vmax):
        print(f"  Advertencia: Después de interpolar, no hay valores válidos para {nombre_empresa}")
        return None
    
    # Crear colormap con transparencia
    base_cmap = colormaps.get_cmap('inferno').resampled(256)
    colors = base_cmap(np.linspace(0, 1, 256))
    colors[0, -1] = 0.0
    cmap_mod = ListedColormap(colors)
    
    # Definir niveles
    vmin_plot = max(vmin, umbral)
    levels = np.linspace(vmin_plot, vmax, 15)
    norm = BoundaryNorm(levels, ncolors=cmap_mod.N, clip=True)
    
    # Crear figura SIN márgenes, ejes ni nada extra
    fig = plt.figure(figsize=(width/100, height/100), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)
    ax.axis('off')
    
    # Graficar solo la superficie
    ax.contourf(
        grid_lon_mesh,
        grid_lat_mesh,
        grid_values_masked,
        levels=levels,
        cmap=cmap_mod,
        norm=norm,
        alpha=0.7
    )
    
    # Guardar con fondo transparente
    plt.savefig(output_png, dpi=100, transparent=True, bbox_inches='tight', pad_inches=0)
    plt.close()
    
    return {
        'success': True,
        'vmin': vmin,
        'vmax': vmax,
        'nombre_empresa': nombre_empresa,
        'clave': clave,
        'es_total': es_total,
        'png_filename': output_png
    }

def create_unified_kml(overlays_info):
    """
    Crea un archivo KML con múltiples GroundOverlays (carpetas para organizar)
    overlays_info: lista de diccionarios con info de cada overlay
    """
    
    # Crear carpetas para organizar las capas
    ground_overlays = []
    
    for info in overlays_info:
        nombre = info['nombre_empresa']
        clave = info['clave']
        vmin = info['vmin']
        vmax = info['vmax']
        es_total = info['es_total']
        png_file = info['png_filename']
        
        # Crear descripción
        if es_total:
            descripcion = f'''<![CDATA[
<h3>Riesgo Total</h3>
<p><b>Valor mínimo:</b> {vmin:.0f}</p>
<p><b>Valor máximo:</b> {vmax:.0f}</p>
<p><i>Nota: Solo se muestran valores mayores a 1</i></p>
]]>'''
        else:
            descripcion = f'''<![CDATA[
<h3>{nombre}</h3>
<p><b>Clave:</b> {clave}</p>
<p><b>Valor mínimo:</b> {vmin:.2e}</p>
<p><b>Valor máximo:</b> {vmax:.2e}</p>
<p><i>Nota: Solo se muestran valores mayores a 0.25</i></p>
]]>'''
        
        # Crear GroundOverlay para esta capa
        overlay = f'''
    <GroundOverlay>
      <name>{nombre}</name>
      <description>{descripcion}</description>
      <Icon>
        <href>{png_file}</href>
      </Icon>
      <LatLonBox>
        <north>{lat_max}</north>
        <south>{lat_min}</south>
        <east>{lon_max}</east>
        <west>{lon_min}</west>
      </LatLonBox>
    </GroundOverlay>'''
        
        ground_overlays.append(overlay)
    
    # Ensamblar KML completo
    kml = f'''<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2">
  <Document>
    <name>Mapas de Riesgo a No cáncer</name>
    <description>Mapas de riesgo para todas sustancias y riesgo total</description>
    
    <Folder>
      <name>Capas de Riesgo</name>
      <description>Activa/desactiva las capas individuales</description>
      {''.join(ground_overlays)}
    </Folder>
  </Document>
</kml>'''
    
    return kml

# --- Generar todas las imágenes overlay ---
print(f"\n{'='*60}")
print(f"Generando archivo KMZ unificado: {output_kmz}")
print(f"Coordenadas: Lon[{lon_min:.6f}, {lon_max:.6f}], Lat[{lat_min:.6f}, {lat_max:.6f}]")
print(f"{'='*60}\n")

overlays_info = []
temp_files = []

for i, empresa in enumerate(cols_empresas):
    print(f"[{i+1}/{len(cols_empresas)}] Procesando {empresa}...")
    
    # Generar nombre único para el PNG
    safe_name = "".join(c for c in empresa.strip() if c.isalnum() or c in ('_',)).rstrip()
    png_filename = f'overlay_{safe_name}.png'
    
    # Generar imagen
    result = generate_overlay_image(empresa, png_filename)
    
    if result:
        overlays_info.append(result)
        temp_files.append(png_filename)
        
        if result['es_total']:
            print(f"  ✓ {result['nombre_empresa']} (min: {result['vmin']:.0f}, max: {result['vmax']:.0f})")
        else:
            print(f"  ✓ {result['nombre_empresa']} [{result['clave']}] (min: {result['vmin']:.2e}, max: {result['vmax']:.2e})")

# --- Crear KML unificado ---
if overlays_info:
    print(f"\nCreando archivo KMZ con {len(overlays_info)} capas...")
    
    kml_content = create_unified_kml(overlays_info)
    
    # Crear archivo KMZ
    try:
        with zipfile.ZipFile(output_kmz, 'w', zipfile.ZIP_DEFLATED) as kmz:
            # Agregar KML
            kmz.writestr('doc.kml', kml_content)
            
            # Agregar todas las imágenes PNG
            for png_file in temp_files:
                if os.path.exists(png_file):
                    kmz.write(png_file, png_file)
        
        print(f"\n{'='*60}")
        print(f"✓ Archivo KMZ unificado creado exitosamente:")
        print(f"  {output_kmz}")
        print(f"  Contiene {len(overlays_info)} capas de riesgo")
        print(f"{'='*60}")
        
        # Limpiar archivos temporales
        print("\nLimpiando archivos temporales...")
        for png_file in temp_files:
            if os.path.exists(png_file):
                os.remove(png_file)
        print("✓ Limpieza completada")
        
    except Exception as e:
        print(f"\n✗ Error al crear KMZ: {e}")
else:
    print("\n✗ No se generaron capas válidas para el KMZ")

print(f"\n{'='*60}")
print("Proceso completado")
print(f"{'='*60}")

## Para empresas

In [ ]:
# Genera UN SOLO archivo KMZ con TODAS las capas de riesgo
# Cada empresa/Total será una capa individual que puede activarse/desactivarse en Google Earth

import pandas as pd
import numpy as np
from pyproj import Transformer
import zipfile
import os
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib import colormaps

# --- Parámetros (deben coincidir con crea_mapa5.py) ---
csv_file = 'riesgosC.csv'
empresas_file = 'empresas.csv'
output_kmz = 'mapas_riesgo_Subs.kmz'

utm_zone = 14
epsg_utm = f"EPSG:326{utm_zone:02d}"
epsg_wgs84 = "EPSG:4326"

# --- Leer datos ---
df = pd.read_csv(csv_file)
df_empresas = pd.read_csv(empresas_file)

# Crear diccionario clave -> nombre
clave_a_nombre = dict(zip(df_empresas['CLAVE'].str.strip(), df_empresas['Nombre'].str.strip()))

# Extraer coordenadas UTM (km a metros)
x_utm = df['x_km'].values * 1000
y_utm = df['y_km'].values * 1000

# Transformar UTM -> WGS84 (lon, lat)
transformer_utm_to_wgs84 = Transformer.from_crs(epsg_utm, epsg_wgs84, always_xy=True)
lon, lat = transformer_utm_to_wgs84.transform(x_utm, y_utm)

# Obtener límites reales de los datos
lon_min, lon_max = lon.min(), lon.max()
lat_min, lat_max = lat.min(), lat.max()

# Columnas de empresas + Total
cols_empresas = [col for col in df.columns if col not in ['x_km', 'y_km', 'lon', 'lat']]

def generate_overlay_image(company_col, output_png, width=2000, height=2000):
    """
    Genera una imagen PNG sin márgenes ni barras de color, solo la superficie de riesgo
    Retorna: (success, vmin, vmax, nombre_empresa, clave, es_total)
    """
    values = df[company_col].values
    
    # Determinar si es Total y obtener nombre
    es_total = company_col.strip().lower() == 'total'
    
    if es_total:
        nombre_empresa = "Riesgo Total"
        clave = None
        umbral = 1.0
    else:
        clave = company_col.strip()
        if clave in clave_a_nombre:
            nombre_empresa = clave_a_nombre[clave]
        else:
            nombre_empresa = clave
            print(f"  Advertencia: Clave '{clave}' no encontrada en empresas.csv")
        umbral = 1e-6
    
    # Convertir a float para poder trabajar con NaN
    values_float = values.astype(float)
    
    # Verificar que hay valores por encima del umbral
    valores_validos = values_float >= umbral
    if not np.any(valores_validos):
        print(f"  Advertencia: No hay valores >= {umbral} para {nombre_empresa}")
        return None
    
    # Crear grilla para interpolación
    num_grid_points = 400
    grid_lon = np.linspace(lon_min, lon_max, num_grid_points)
    grid_lat = np.linspace(lat_min, lat_max, num_grid_points)
    grid_lon_mesh, grid_lat_mesh = np.meshgrid(grid_lon, grid_lat)
    
    # Interpolación con TODOS los valores
    grid_values = griddata(
        points=(lon, lat),
        values=values_float,
        xi=(grid_lon_mesh, grid_lat_mesh),
        method='nearest'
    )
    
    # Aplicar máscara para valores bajo el umbral
    grid_values_masked = grid_values.copy()
    grid_values_masked[grid_values < umbral] = np.nan
    
    # Verificar valores válidos después de interpolar
    vmin = np.nanmin(grid_values_masked)
    vmax = np.nanmax(grid_values_masked)
    
    if np.isnan(vmin) or np.isnan(vmax):
        print(f"  Advertencia: Después de interpolar, no hay valores válidos para {nombre_empresa}")
        return None
    
    # Crear colormap con transparencia
    base_cmap = colormaps.get_cmap('inferno').resampled(256)
    colors = base_cmap(np.linspace(0, 1, 256))
    colors[0, -1] = 0.0
    cmap_mod = ListedColormap(colors)
    
    # Definir niveles
    vmin_plot = max(vmin, umbral)
    levels = np.linspace(vmin_plot, vmax, 15)
    norm = BoundaryNorm(levels, ncolors=cmap_mod.N, clip=True)
    
    # Crear figura SIN márgenes, ejes ni nada extra
    fig = plt.figure(figsize=(width/100, height/100), dpi=100)
    ax = fig.add_axes([0, 0, 1, 1])
    ax.set_xlim(lon_min, lon_max)
    ax.set_ylim(lat_min, lat_max)
    ax.axis('off')
    
    # Graficar solo la superficie
    ax.contourf(
        grid_lon_mesh,
        grid_lat_mesh,
        grid_values_masked,
        levels=levels,
        cmap=cmap_mod,
        norm=norm,
        alpha=0.7
    )
    
    # Guardar con fondo transparente
    plt.savefig(output_png, dpi=100, transparent=True, bbox_inches='tight', pad_inches=0)
    plt.close()
    
    return {
        'success': True,
        'vmin': vmin,
        'vmax': vmax,
        'nombre_empresa': nombre_empresa,
        'clave': clave,
        'es_total': es_total,
        'png_filename': output_png
    }

def create_unified_kml(overlays_info):
    """
    Crea un archivo KML con múltiples GroundOverlays (carpetas para organizar)
    overlays_info: lista de diccionarios con info de cada overlay
    """
    
    # Crear carpetas para organizar las capas
    ground_overlays = []
    
    for info in overlays_info:
        nombre = info['nombre_empresa']
        clave = info['clave']
        vmin = info['vmin']
        vmax = info['vmax']
        es_total = info['es_total']
        png_file = info['png_filename']
        
        # Crear descripción
        if es_total:
            descripcion = f'''<![CDATA[
<h3>Riesgo Total</h3>
<p><b>Valor mínimo:</b> {vmin:.0f}</p>
<p><b>Valor máximo:</b> {vmax:.0f}</p>
<p><i>Nota: Solo se muestran valores mayores a 1</i></p>
]]>'''
        else:
            descripcion = f'''<![CDATA[
<h3>{nombre}</h3>
<p><b>Clave:</b> {clave}</p>
<p><b>Valor mínimo:</b> {vmin:.2e}</p>
<p><b>Valor máximo:</b> {vmax:.2e}</p>
<p><i>Nota: Solo se muestran valores mayores a 1e-6</i></p>
]]>'''
        
        # Crear GroundOverlay para esta capa
        overlay = f'''
    <GroundOverlay>
      <name>{nombre}</name>
      <description>{descripcion}</description>
      <Icon>
        <href>{png_file}</href>
      </Icon>
      <LatLonBox>
        <north>{lat_max}</north>
        <south>{lat_min}</south>
        <east>{lon_max}</east>
        <west>{lon_min}</west>
      </LatLonBox>
    </GroundOverlay>'''
        
        ground_overlays.append(overlay)
    
    # Ensamblar KML completo
    kml = f'''<?xml version="1.0" encoding="UTF-8"?>
<kml xmlns="http://www.opengis.net/kml/2.2">
  <Document>
    <name>Mapas de Riesgo a Cáncer</name>
    <description>Mapas de riesgo para todas las empresas y riesgo total</description>
    
    <Folder>
      <name>Capas de Riesgo</name>
      <description>Activa/desactiva las capas individuales</description>
      {''.join(ground_overlays)}
    </Folder>
  </Document>
</kml>'''
    
    return kml

# --- Generar todas las imágenes overlay ---
print(f"\n{'='*60}")
print(f"Generando archivo KMZ unificado: {output_kmz}")
print(f"Coordenadas: Lon[{lon_min:.6f}, {lon_max:.6f}], Lat[{lat_min:.6f}, {lat_max:.6f}]")
print(f"{'='*60}\n")

overlays_info = []
temp_files = []

for i, empresa in enumerate(cols_empresas):
    print(f"[{i+1}/{len(cols_empresas)}] Procesando {empresa}...")
    
    # Generar nombre único para el PNG
    safe_name = "".join(c for c in empresa.strip() if c.isalnum() or c in ('_',)).rstrip()
    png_filename = f'overlay_{safe_name}.png'
    
    # Generar imagen
    result = generate_overlay_image(empresa, png_filename)
    
    if result:
        overlays_info.append(result)
        temp_files.append(png_filename)
        
        if result['es_total']:
            print(f"  ✓ {result['nombre_empresa']} (min: {result['vmin']:.0f}, max: {result['vmax']:.0f})")
        else:
            print(f"  ✓ {result['nombre_empresa']} [{result['clave']}] (min: {result['vmin']:.2e}, max: {result['vmax']:.2e})")

# --- Crear KML unificado ---
if overlays_info:
    print(f"\nCreando archivo KMZ con {len(overlays_info)} capas...")
    
    kml_content = create_unified_kml(overlays_info)
    
    # Crear archivo KMZ
    try:
        with zipfile.ZipFile(output_kmz, 'w', zipfile.ZIP_DEFLATED) as kmz:
            # Agregar KML
            kmz.writestr('doc.kml', kml_content)
            
            # Agregar todas las imágenes PNG
            for png_file in temp_files:
                if os.path.exists(png_file):
                    kmz.write(png_file, png_file)
        
        print(f"\n{'='*60}")
        print(f"✓ Archivo KMZ unificado creado exitosamente:")
        print(f"  {output_kmz}")
        print(f"  Contiene {len(overlays_info)} capas de riesgo")
        print(f"{'='*60}")
        
        # Limpiar archivos temporales
        print("\nLimpiando archivos temporales...")
        for png_file in temp_files:
            if os.path.exists(png_file):
                os.remove(png_file)
        print("✓ Limpieza completada")
        
    except Exception as e:
        print(f"\n✗ Error al crear KMZ: {e}")
else:
    print("\n✗ No se generaron capas válidas para el KMZ")

print(f"\n{'='*60}")
print("Proceso completado")
print(f"{'='*60}")